In [1]:
!pip install yfinance pandas pyarrow requests_cache
!pip install google-cloud-storage -q
!pip install arch
!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.4 MB/s eta 0:00:00


In [2]:
import yfinance as yf
import pandas as pd
from pathlib import Path
from arch import arch_model
from matplotlib import pyplot as plt
import requests_cache
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
PROJECT_DIR = "/content/drive/MyDrive/vol_forecasting"

In [5]:
def get_sp500_tickers(n=50):
    import urllib.request
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    req = urllib.request.Request(url, headers=headers)
    response = urllib.request.urlopen(req)

    tables = pd.read_html(response)
    tickers = tables[0]["Symbol"].tolist()
    tickers = [t.replace(".", "-") for t in tickers]
    return tickers[:n]

tickers = get_sp500_tickers(n=50)
print(f"First 5: {tickers[:5]}")
print(f"Total: {len(tickers)}")


raw = yf.download(
    tickers=tickers,
    start="2018-01-01",
    end="2024-12-31",
    auto_adjust=True,
    progress=True,
)
vix = yf.download(
    tickers="^VIX",
    start="2018-01-01",
    end="2024-12-31",
    auto_adjust=True,
    progress=True,
)

print(f"Downloaded shape: {raw.shape}")


First 5: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']
Total: 50


[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  1 of 1 completed

Downloaded shape: (1760, 250)


In [6]:
vix

Price,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,
2018-01-02,9.770000,11.070000,9.520000,10.950000,0
2018-01-03,9.150000,9.650000,8.940000,9.560000,0
2018-01-04,9.220000,9.310000,8.920000,9.010000,0
2018-01-05,9.220000,9.540000,9.000000,9.100000,0
2018-01-08,9.520000,9.890000,9.320000,9.610000,0
...,...,...,...,...,...
2024-12-23,16.780001,20.020000,16.740000,18.090000,0
2024-12-24,14.270000,17.040001,14.270000,16.969999,0


In [7]:
frames = []

for ticker in tickers:
    try:
        df = raw.xs(ticker, axis=1, level=1).copy()
        df["ticker"] = ticker
        df.index.name = "date"
        frames.append(df)
    except KeyError:
        print(f"Skipping {ticker} — no data")

long_df = pd.concat(frames).reset_index()
long_df.columns = long_df.columns.str.lower()
long_df = long_df.dropna(subset=["close"])
long_df = long_df.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Shape: {long_df.shape}")
print(f"Tickers: {long_df['ticker'].nunique()}")
print(long_df.head())



Shape: (86433, 7)
Tickers: 50
Price       date      close       high        low       open     volume ticker
0     2018-01-02  63.522751  63.795261  63.278431  63.353607  1047800.0      A
1     2018-01-03  65.139038  65.298783  63.522777  63.541575  1698900.0      A
2     2018-01-04  64.650391  65.608867  64.631593  65.345756  2230700.0      A
3     2018-01-05  65.684036  65.871971  64.584606  64.584606  1632500.0      A
4     2018-01-08  65.825012  66.088123  65.355169  65.524313  1613400.0      A


In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
from arch import arch_model

# 1. Create a bucket to store each finished ticker
all_processed_data = []

for symbol in tickers:
    print(f"--- Processing {symbol} ---")

    # Download data for this ticker
    try:
        df = yf.download(symbol, start="2018-01-01", progress=True)
        if df.empty:
            print(f"Skipping {symbol}: No data found.")
            continue

        # Clean up columns if yfinance returns multi-index
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # --- BASIC FEATURES ---
        df["log_ret"] = np.log(df["Close"] / df["Close"].shift(1)) * 100
        df["sqr_ret"] = df["log_ret"]**2
        df["real_vol"] = df["log_ret"].rolling(10).std() * np.sqrt(252)

        # EWMA Vol
        df["EWMA"] = df["sqr_ret"].ewm(span=20, adjust=False).mean()
        df["ewma_vol"] = np.sqrt(df["EWMA"])

        # --- CYCLIC ENCODINGS (For TFT) ---
        df['day_of_week'] = df.index.dayofweek
        df['month'] = df.index.month
        df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 5)
        df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 5)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

        # --- LAG FEATURES & VIX ---
        df["vol_lag1"] = df["real_vol"].shift(1)
        # Ensure VIX aligns with the current ticker's dates
        df["vix"] = vix["Close"].reindex(df.index).ffill()

        # --- GARCH ROLLING FORECAST ---
        rolling_p = []
        # We only forecast if we have enough data
        if len(df) > 365:
            for i in range(365):
                train_slice = df["log_ret"].iloc[:-(365-i)].dropna()
                try:
                    m = arch_model(train_slice, p=1, q=1, rescale=False)
                    res = m.fit(disp='off')
                    pred = res.forecast(horizon=1).variance.values[-1, 0]
                    rolling_p.append(np.sqrt(pred))
                except:
                    rolling_p.append(np.nan)

            df["garch_vol"] = [np.nan] * (len(df) - 365) + rolling_p
        else:
            df["garch_vol"] = np.nan

        # --- PREPARE FOR COLLECTION ---
        df['symbol'] = symbol
        # Reset index so 'Date' becomes a column (important for TFT time_idx creation later)
        all_processed_data.append(df.reset_index())

    except Exception as e:
        print(f"Error processing {symbol}: {e}")

# --- FINAL SAVE (OUTSIDE THE LOOP) ---
if all_processed_data:
    final_df = pd.concat(all_processed_data, ignore_index=True)

    # Save as a single unified file to avoid Google Drive sync lag
    final_df.to_parquet('unified_volatility_data.parquet', engine='pyarrow')

    print("-" * 30)
    print(f"Success! {final_df['symbol'].nunique()} tickers processed.")
    print(f"Total rows saved: {len(final_df)}")
    print(f"Data saved to: unified_volatility_data.parquet")
else:
    print("No data was processed.")

/tmp/ipykernel_1498/1895163402.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2018-01-01", progress=True)


--- Processing MMM ---


[*********************100%***********************]  1 of 1 completed


--- Processing AOS ---


/tmp/ipykernel_1498/1895163402.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2018-01-01", progress=True)
[*********************100%***********************]  1 of 1 completed


...

--- Processing T ---


/tmp/ipykernel_1498/1895163402.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2018-01-01", progress=True)
[*********************100%***********************]  1 of 1 completed


--- Processing ATO ---


/tmp/ipykernel_1498/1895163402.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start="2018-01-01", progress=True)
[*********************100%***********************]  1 of 1 completed


------------------------------
Success! 50 tickers processed.
Total rows saved: 104083
Data saved to: unified_volatility_data.parquet


In [9]:
from google.colab import drive
drive.mount('/content/drive')

# Then save your parquet there
df.to_parquet('/content/drive/MyDrive/unified_volatility_data.parquet', engine='pyarrow')
print("Successfully saved as a single file!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully saved as a single file!


In [ ]:

all_tickers = df['symbol'].unique()
print(all_tickers)

In [ ]:

df = df.dropna()
outliers = df[df['real_vol'] > 50]
print(len(outliers))

PreProcessing


In [ ]:
df['symbol'] = df['symbol'].astype(str)
df["time_idx"] = df.groupby("symbol").cumcount()
counts = df['symbol'].value_counts()
valid_tickers = counts[counts > 500].index
df = df[df['symbol'].isin(valid_tickers)]
df.to_parquet('tft_model_input.parquet', partition_cols=['symbol'])

In [ ]:
df_loaded = pd.read_parquet('/content/drive/MyDrive/processed_volatility_data')
print(f"Loaded shape: {df_loaded.shape}")
print(df_loaded.dtypes)

In [ ]:
def get_ticker_metadata():
    import urllib.request
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    req = urllib.request.Request(url, headers=headers)
    response = urllib.request.urlopen(req)

    table = pd.read_html(response)[0]

    # Keep only the columns we care about
    meta = table[["Symbol", "Security", "GICS Sector"]].copy()
    meta.columns = ["ticker", "company", "sector"]
    meta["ticker"] = meta["ticker"].str.replace(".", "-", regex=False)

    return meta
meta = get_ticker_metadata()
print(meta.head(10))

In [ ]:
df = pd.read_parquet(f"{PROJECT_DIR}/ohlcv.parquet")
df = df.merge(meta, on="ticker", how="left")

# Confirm it worked
print("\nMissing company names:", df["company"].isna().sum())

In [ ]:
df

In [ ]:
print("Columns:", df.columns.tolist())
print("Tickers:", df['ticker'].nunique())
print("\nMissing values:\n", df.isna().sum())
print("\nSample:\n", df.head(10))

Implementing GARCH


In [ ]:
att = df[df["ticker"] == "T"].copy()
att

In [ ]:
returns = att.close.pct_change().dropna()
plt.figure(figsize = (10,4))
plt.plot(returns)

In [ ]:
model = arch_model(returns,p = 1,q = 1)
model_fit = model.fit()

In [ ]:
model_fit.summary()

In [ ]:
returns

Rolling volatility prediction using the GARCH(1,1) looking at the last 365 days and comparison


In [ ]:
rolling_p = []
for i in range(365):
  train = att[:-(365-i)]
  returns = train.close.pct_change().dropna()
  model = arch_model(returns,p = 1,q = 1)
  model_fit = model.fit()
  pred = model_fit.forecast(horizon = 1)
  rolling_p.append(np.sqrt(pred.variance.values[-1,0]))


In [ ]:
att["log_ret"] = np.log(att["close"]/(att["close"].shift(1)))
att["returns"] = att["close"]/(att["close"].shift(1))
att["real_vol"] = att["log_ret"].rolling(10).std()*np.sqrt(252)
att["sqr_returns"] = att["log_ret"]**2
att["EWMA"] = att["sqr_returns"].ewm(span = 20,adjust = False).mean()
att["ewma_vol"] = np.sqrt(att["EWMA"])
SPLIT_DATE = "2022-01-01"
train = att[att["date"]<=SPLIT_DATE]
test = att[att["date"]>SPLIT_DATE]

In [ ]:
rolling_p = np.array(rolling_p)
rolling_p = pd.Series(rolling_p, index = test.index[-365:])
rolling_p

In [ ]:
volatility = att["real_vol"]
plt.figure(figsize=(12, 6))
plt.plot(test.index[-365:], test[-365:]["real_vol"]/np.sqrt(252)*100, label="Realized Volatility (Actual)",
         color="blue", alpha=0.4)
plt.plot(test.index[-365:], np.array(rolling_p)*100, label="Predicted Volatility (Rolling)",
         color="red", alpha=0.4)
plt.legend()

In [ ]:
test  = test.copy()
test["Random walk"] = test["real_vol"].shift(1)
test

In [ ]:
plt.figure(figsize=(15, 8))

# 1. Actual Realized Volatility (Smoothed by 20-day window)
plt.plot(test.index[-365:], test["real_vol"].iloc[-365:]/np.sqrt(252)*100,
         label="Realized Vol (Actual)", color="blue", alpha=0.3, linestyle='--')

# 2. EWMA (Exponentially Weighted)
plt.plot(test.index[-365:], test["ewma_vol"].iloc[-365:]*100,
         label="EWMA Volatility", color="green", alpha=0.7)

# 3. GARCH Predictions
plt.plot(test.index[-365:], np.array(rolling_p)*100,
         label="GARCH(1,1) Forecast", color="red", alpha=0.8)

plt.title("Comparison of Volatility Models: AT&T", fontsize=16)
plt.ylabel("Daily Volatility (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()